# 第 57 章：Capstone 校友案例与导师接力

这个 notebook 用一个极小的 mentor-relay table，对比“只按 continuity 分数排序”的 naive baseline 和“先检查隐私边界、mentor 可用性、范围与案例准备”的导师接力 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_alumni_mentor_relay_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['continuity_score'] = float(row['continuity_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} mentor-relay items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Example readiness:', ordered_counts(rows, 'example_readiness_status'))
print('Privacy boundary:', ordered_counts(rows, 'privacy_boundary_status'))
print('Mentor availability:', ordered_counts(rows, 'mentor_availability_status'))
print('Handoff scope:', ordered_counts(rows, 'handoff_scope_status'))
print('Owner status:', ordered_counts(rows, 'owner_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['relay_id']
    for row in rows
    if row['reference_route'] == 'mentor_relay_ready'
)
top4_continuity = sorted(
    rows,
    key=lambda row: row['continuity_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'mentor_relay_ready'
    for row in top4_continuity
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'mentor_relay_ready'
    for row in top4_continuity
)
baseline_ready_precision = baseline_ready_hits / len(top4_continuity)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_continuity)

print('Reference mentor-relay-ready items:', reference_ready_ids)
print('Top-4 by continuity score:')
for row in top4_continuity:
    print(
        f"  {row['relay_id']}: continuity={row['continuity_score']:.2f}, "
        f"route={row['reference_route']}, area={row['relay_area']}"
    )
print('Baseline relay precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_relay_item(row):
    if row['privacy_boundary_status'] != 'clear':
        return 'review_privacy_boundary', 'student work or example boundary still needs privacy review'
    if row['mentor_availability_status'] != 'confirmed':
        return 'confirm_mentor_availability', 'mentor time or commitment is not confirmed yet'
    if row['handoff_scope_status'] != 'bounded':
        return 'narrow_handoff_scope', 'relay scope is too broad and needs tightening'
    if row['example_readiness_status'] != 'ready':
        return 'prepare_example_for_relay', 'example is not ready to be used for relay yet'
    return 'mentor_relay_ready', 'example is ready, boundary is clear, and mentor support is bounded and available'


routed_rows = []
for row in rows:
    route, reason = route_relay_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Mentor-relay workflow routes:')
for row in routed_rows:
    print(
        f"  {row['relay_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'mentor_relay_ready']
example_queue = [row for row in routed_rows if row['route'] == 'prepare_example_for_relay']
availability_queue = [row for row in routed_rows if row['route'] == 'confirm_mentor_availability']
scope_queue = [row for row in routed_rows if row['route'] == 'narrow_handoff_scope']
privacy_queue = [row for row in routed_rows if row['route'] == 'review_privacy_boundary']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'mentor_relay_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Relay-ready queue:', [row['relay_id'] for row in ready_queue])
print('Prepare-example queue:', [row['relay_id'] for row in example_queue])
print('Confirm-availability queue:', [row['relay_id'] for row in availability_queue])
print('Narrow-scope queue:', [row['relay_id'] for row in scope_queue])
print('Review-privacy queue:', [row['relay_id'] for row in privacy_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured relay precision:', round(ready_precision, 3))


In [ ]:
def relay_steps(row):
    steps = []
    if row['privacy_boundary_status'] != 'clear':
        steps.append('先复核学生作品、示例截图和展示材料的隐私边界。')
    if row['mentor_availability_status'] != 'confirmed':
        steps.append('确认 mentor 的时间窗口、答疑方式和投入上限。')
    if row['handoff_scope_status'] != 'bounded':
        steps.append('收窄接力范围，明确 mentor 只支持哪些问题和阶段。')
    if row['example_readiness_status'] != 'ready':
        steps.append('把案例整理成下一届可展示、可理解、可复用的最小版本。')
    return steps or ['可以进入 mentor relay package；发布时保留边界说明和 contact note。']


for row in routed_rows:
    if row['route'] != 'mentor_relay_ready':
        print(f"\n{row['relay_id']} -> {row['route']} ({row['relay_area']})")
        for step in relay_steps(row):
            print(' -', step)


In [ ]:
relay_outline = [
    'Example pack: which alumni projects are safe and useful to show next cohort',
    'Mentor role: experience share, project advice, short office hour, or one-time panel',
    'Time boundary: what mentors can realistically commit to',
    'Student expectation note: mentor support is guidance, not project outsourcing',
    'Privacy note: what can be shown publicly, internally, or only after approval',
    'Owner path: who maintains the relay package and refreshes it next cycle',
]

relay_assistant_prompt = '''你是我的 capstone 校友接力助手。

任务：
1. 先阅读 mentor-relay table，不要只按 continuity 排序；
2. 先检查 privacy boundary；
3. 再检查 mentor availability、handoff scope 和 example readiness；
4. 把每个条目 route 到 mentor_relay_ready、prepare_example_for_relay、
   confirm_mentor_availability、narrow_handoff_scope 或 review_privacy_boundary；
5. 对每个非 ready 条目输出 relay 前 checklist。

输出格式：
- Relay-ready items
- Prepare-example items
- Confirm-availability items
- Narrow-scope items
- Review-privacy items
- Relay checklist by item
'''

print('Mentor relay outline:')
for item in relay_outline:
        print(' -', item)

print('\nAssistant prompt:')
print(relay_assistant_prompt)


In [ ]:
relay_snapshot = {
    'dataset': DATA_PATH.name,
    'n_relay_items': len(rows),
    'baseline_top4_relay_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'mentor_relay_ready': [row['relay_id'] for row in ready_queue],
    'prepare_example_for_relay': [row['relay_id'] for row in example_queue],
    'confirm_mentor_availability': [row['relay_id'] for row in availability_queue],
    'narrow_handoff_scope': [row['relay_id'] for row in scope_queue],
    'review_privacy_boundary': [row['relay_id'] for row in privacy_queue],
    'python_version': platform.python_version(),
}

print('Mentor relay snapshot:')
for key, value in relay_snapshot.items():
    print(f'  {key}: {value}')
